# Smart Parking — Component Benchmark

Test từng thành phần AI riêng biệt, đo latency / accuracy / throughput.

### Components:
1. **PlateDetector** — YOLOv8n plate detection (data riêng)
2. **PlateOCR + Validator** — PaddleOCR recognition (data riêng, .txt ground truth)
3. **FaceEngine** — InsightFace detection + embedding (LFW dataset)
4. **ParkingDB** — pgvector cosine search (dùng embedding thật từ LFW)
5. **End-to-End Pipeline** — Full flow benchmark

In [2]:
import numpy as np
import cv2
import time
import os
import sys
import glob
import json
import logging
import random
from pathlib import Path
from collections import defaultdict, Counter

%matplotlib inline
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.ticker import MaxNLocator

plt.style.use('dark_background')
plt.rcParams.update({
    'figure.figsize': (12, 5),
    'figure.dpi': 100,
    'axes.grid': True,
    'grid.alpha': 0.15,
    'font.size': 11,
})

C = {
    'green': '#22c55e', 'blue': '#3b82f6', 'yellow': '#eab308',
    'red': '#ef4444', 'purple': '#a855f7', 'cyan': '#06b6d4',
    'orange': '#f97316', 'pink': '#ec4899',
}

logging.basicConfig(level=logging.WARNING)
print('Imports OK')

Imports OK


In [12]:
PROJECT_DIR = Path('/home/somethink/parking_system')
sys.path.insert(0, str(PROJECT_DIR))

# ── 1) PLATE DETECTION DATA ──
DET_IMAGES_DIR = 'yolo_plate_data/images/val'
DET_LABELS_DIR = 'yolo_plate_data/labels/val'

# ── 2) PLATE RECOGNITION (OCR) DATA ──
OCR_DATA_DIR = ''

# ── 3) FACE DATA ──
LFW_ROOT = str(PROJECT_DIR / 'benchmark_data' / 'lfw_funneled')

# ╔══════════════════════════════════════════════════════════════╗
# ║  MODEL COMPARISON — thêm model mới vào đây để so sánh      ║
# ║  Mỗi entry là 1 dict {name, path, ...}                     ║
# ║  Entry đầu tiên = baseline, các entry sau = challenger      ║
# ╚══════════════════════════════════════════════════════════════╝

# Plate Detector models — thêm/bớt tùy ý
# Mỗi entry: name (hiển thị), path (file .engine/.pt), imgsz, conf
# ╔══════════════════════════════════════════════════════════════════╗
# ║  PLATE DETECTION MODELS — so sánh bất kỳ architecture nào     ║
# ║                                                                ║
# ║  type:                                                         ║
# ║    'ultralytics' → YOLOv5/v8/v9/v10/v11, RT-DETR (.pt/.engine)║
# ║    'yolov5'      → YOLOv5 gốc (torch.hub)                     ║
# ║    'onnx'        → Bất kỳ model ONNX nào                      ║
# ║    'custom'      → Tự viết class, truyền vào callable          ║
# ║                                                                ║
# ║  Tất cả đều trả về: [{bbox:(x1,y1,x2,y2), conf:float}]       ║
# ╚══════════════════════════════════════════════════════════════════╝

PLATE_DET_MODELS = [
    # ── Baseline: YOLOv8n TensorRT ──
    {
        'name': 'YOLOv8n TRT',
        'type': 'ultralytics',
        'path': '/home/somethink/parking_system/models/plate_yolov8n.engine',
        'imgsz': 640,
        'conf': 0.4,
    },

    # ── Uncomment để so sánh ──

    # # YOLOv8 sizes khác
    # {'name': 'YOLOv8s', 'type': 'ultralytics',
    #  'path': '/path/to/yolov8s_plate.pt', 'imgsz': 640, 'conf': 0.4},
    # {'name': 'YOLOv8m', 'type': 'ultralytics',
    #  'path': '/path/to/yolov8m_plate.pt', 'imgsz': 640, 'conf': 0.4},

    # # YOLOv5 (qua ultralytics — hỗ trợ load .pt YOLOv5)
    # {'name': 'YOLOv5n', 'type': 'ultralytics',
    #  'path': '/path/to/yolov5n_plate.pt', 'imgsz': 640, 'conf': 0.4},
    # {'name': 'YOLOv5s', 'type': 'ultralytics',
    #  'path': '/path/to/yolov5s_plate.pt', 'imgsz': 640, 'conf': 0.4},

    # # YOLOv5 gốc (torch.hub — nếu model train bằng ultralytics/yolov5 repo)
    # {'name': 'YOLOv5s (hub)', 'type': 'yolov5',
    #  'path': '/path/to/yolov5s_plate.pt', 'imgsz': 640, 'conf': 0.4},

    # # YOLOv9
    # {'name': 'YOLOv9c', 'type': 'ultralytics',
    #  'path': '/path/to/yolov9c_plate.pt', 'imgsz': 640, 'conf': 0.4},

    # # YOLOv10
    # {'name': 'YOLOv10n', 'type': 'ultralytics',
    #  'path': '/path/to/yolov10n_plate.pt', 'imgsz': 640, 'conf': 0.4},

    # # YOLO11 (ultralytics mới nhất)
    # {'name': 'YOLO11n', 'type': 'ultralytics',
    #  'path': '/path/to/yolo11n_plate.pt', 'imgsz': 640, 'conf': 0.4},

    # # RT-DETR (transformer-based, qua ultralytics)
    # {'name': 'RT-DETR-l', 'type': 'ultralytics',
    #  'path': '/path/to/rtdetr_l_plate.pt', 'imgsz': 640, 'conf': 0.4},

    # # ONNX Runtime (bất kỳ model nào export sang .onnx)
    # {'name': 'YOLOv8n ONNX', 'type': 'onnx',
    #  'path': '/path/to/plate_det.onnx', 'imgsz': 640, 'conf': 0.4},
]


OCR_MODELS = [
    {'name': 'PaddleOCR (current)', 'lang': 'en', 'use_gpu': True},
    # {'name': 'PaddleOCR ch', 'lang': 'ch', 'use_gpu': True},
]

# Face models — thêm/bớt tùy ý
FACE_MODELS = [
    {'name': 'buffalo_sc (current)', 'model_pack': 'buffalo_sc', 'det_size': (640, 640)},
    # {'name': 'buffalo_l (larger)', 'model_pack': 'buffalo_l', 'det_size': (640, 640)},
    # {'name': 'buffalo_sc 320', 'model_pack': 'buffalo_sc', 'det_size': (320, 320)},
]

# Benchmark params
N_WARMUP = 5
N_BENCH  = 100

# ── Validate paths ──
def check_dir(name, path):
    if not path:
        print(f'  ⚠️  {name}: CHƯA SET'); return False
    if Path(path).is_dir():
        n = len(list(Path(path).iterdir()))
        print(f'  ✅ {name}: {path} ({n} files)'); return True
    print(f'  ❌ {name}: KHÔNG TỒN TẠI'); return False

print('Paths:')
HAS_DET_IMG = check_dir('Det images', DET_IMAGES_DIR)
HAS_DET_LBL = check_dir('Det labels', DET_LABELS_DIR)
HAS_DET = HAS_DET_IMG and HAS_DET_LBL
HAS_OCR = check_dir('OCR data', OCR_DATA_DIR)
HAS_LFW = check_dir('LFW faces', LFW_ROOT)

print(f'\nModels to compare:')
print(f'  Plate Det: {len(PLATE_DET_MODELS)} model(s)')
for m in PLATE_DET_MODELS:
    exists = os.path.exists(m['path'])
    print(f'    {"✅" if exists else "❌"} {m["name"]}: {m["path"]}')
print(f'  OCR:       {len(OCR_MODELS)} config(s)')
print(f'  Face:      {len(FACE_MODELS)} model(s)')

Checking paths:
  ✅ Det images: yolo_plate_data/images/val (1651 files)
  ✅ Det labels: yolo_plate_data/labels/val (1651 files)
  ⚠️  OCR data: CHƯA SET
  ❌ LFW faces: KHÔNG TỒN TẠI
  ✅ Plate model: /home/somethink/parking_system/models/plate_yolov8n.engine


In [4]:
def bench(func, n=N_BENCH, warmup=N_WARMUP, label=''):
    """Benchmark 1 function → dict latency stats."""
    for _ in range(warmup):
        func()
    times = []
    for _ in range(n):
        t0 = time.perf_counter()
        func()
        times.append((time.perf_counter() - t0) * 1000)
    arr = np.array(times)
    s = {
        'label': label, 'n': n,
        'mean': arr.mean(), 'std': arr.std(),
        'min': arr.min(), 'max': arr.max(),
        'p50': np.percentile(arr, 50),
        'p95': np.percentile(arr, 95),
        'p99': np.percentile(arr, 99),
        'fps': 1000.0 / arr.mean() if arr.mean() > 0 else 0,
        'raw': arr,
    }
    print(f'  {label:30s}  avg={s["mean"]:7.2f}ms  '
          f'p95={s["p95"]:7.2f}ms  fps={s["fps"]:6.1f}')
    return s

def iou_calc(a, b):
    x1=max(a[0],b[0]); y1=max(a[1],b[1])
    x2=min(a[2],b[2]); y2=min(a[3],b[3])
    inter = max(0,x2-x1)*max(0,y2-y1)
    union = (a[2]-a[0])*(a[3]-a[1])+(b[2]-b[0])*(b[3]-b[1])-inter
    return inter/max(union,1e-6)

def edit_distance(s1, s2):
    m, n = len(s1), len(s2)
    dp = [[0]*(n+1) for _ in range(m+1)]
    for i in range(m+1): dp[i][0] = i
    for j in range(n+1): dp[0][j] = j
    for i in range(1, m+1):
        for j in range(1, n+1):
            dp[i][j] = min(dp[i-1][j]+1, dp[i][j-1]+1,
                           dp[i-1][j-1]+(0 if s1[i-1]==s2[j-1] else 1))
    return dp[m][n]


def pad_to_square(img):
    """
    Pad ảnh về hình vuông (max dimension).
    TensorRT engine export 640x640 cố định — nếu truyền ảnh
    không vuông, ultralytics letterbox thành shape lệch (vd 416x640)
    → TRT reject. Pad trước để YOLO resize về đúng 640x640.
    """
    h, w = img.shape[:2]
    if h == w:
        return img
    size = max(h, w)
    padded = np.zeros((size, size, 3), dtype=np.uint8)
    padded[:h, :w] = img
    return padded




# ═══════════════════════════════════════
# Universal Detector — load bất kỳ model
# ═══════════════════════════════════════

class UniversalDetector:
    """
    Wrapper thống nhất cho mọi object detection model.
    Tất cả trả về: [{bbox: (x1,y1,x2,y2), conf: float}]
    
    Supported types:
      ultralytics → YOLOv5/v8/v9/v10/v11, RT-DETR, YOLO-NAS
      yolov5      → YOLOv5 gốc qua torch.hub
      onnx        → ONNX Runtime (cần tự parse output)
      custom      → Bất kỳ callable(frame) → [{bbox, conf}]
    """
    
    def __init__(self, cfg: dict):
        self.name = cfg['name']
        self.model_type = cfg.get('type', 'ultralytics')
        self.imgsz = cfg.get('imgsz', 640)
        self.conf = cfg.get('conf', 0.4)
        self.model = None
        
        path = cfg.get('path', '')
        
        if self.model_type == 'ultralytics':
            self._load_ultralytics(path)
        elif self.model_type == 'yolov5':
            self._load_yolov5_hub(path)
        elif self.model_type == 'onnx':
            self._load_onnx(path)
        elif self.model_type == 'custom':
            # cfg['callable'] phải là function(frame) → [{bbox, conf}]
            self.model = cfg['callable']
        else:
            raise ValueError(f"Unknown model type: {self.model_type}")
    
    def _load_ultralytics(self, path):
        """Load YOLOv5/v8/v9/v10/v11/RT-DETR qua ultralytics."""
        from ultralytics import YOLO
        
        # Auto TRT export nếu là .pt và chưa có .engine
        engine_path = path.rsplit('.', 1)[0] + '.engine'
        if path.endswith('.pt') and not os.path.exists(engine_path):
            print(f'    Exporting {self.name} → TensorRT...')
            YOLO(path).export(format='engine', half=True,
                              imgsz=self.imgsz, device=0)
            path = engine_path
        elif os.path.exists(engine_path) and path.endswith('.pt'):
            path = engine_path
        
        self.model = YOLO(path, task='detect')
        # Warmup
        self.model(np.zeros((self.imgsz, self.imgsz, 3), dtype=np.uint8),
                   verbose=False)
    
    def _load_yolov5_hub(self, path):
        """Load YOLOv5 gốc qua torch.hub."""
        import torch
        self.model = torch.hub.load('ultralytics/yolov5', 'custom',
                                     path=path, force_reload=False)
        self.model.conf = self.conf
        self.model.iou = 0.45
    
    def _load_onnx(self, path):
        """Load ONNX model."""
        import onnxruntime as ort
        providers = ['TensorrtExecutionProvider',
                     'CUDAExecutionProvider',
                     'CPUExecutionProvider']
        self.model = ort.InferenceSession(path, providers=providers)
        self._onnx_input_name = self.model.get_inputs()[0].name
    
    def __call__(self, frame: np.ndarray) -> list:
        if self.model_type == 'ultralytics':
            return self._infer_ultralytics(frame)
        elif self.model_type == 'yolov5':
            return self._infer_yolov5(frame)
        elif self.model_type == 'onnx':
            return self._infer_onnx(frame)
        elif self.model_type == 'custom':
            return self.model(frame)
        return []
    
    def _infer_ultralytics(self, frame):
        results = self.model(frame, imgsz=self.imgsz, conf=self.conf,
                             verbose=False)
        out = []
        for r in results:
            for box in r.boxes:
                x1, y1, x2, y2 = box.xyxy[0].int().tolist()
                out.append({'bbox': (x1, y1, x2, y2),
                            'conf': float(box.conf)})
        return out
    
    def _infer_yolov5(self, frame):
        results = self.model(frame, size=self.imgsz)
        out = []
        for *xyxy, conf, cls in results.xyxy[0].cpu().numpy():
            x1, y1, x2, y2 = map(int, xyxy)
            out.append({'bbox': (x1, y1, x2, y2), 'conf': float(conf)})
        return out
    
    def _infer_onnx(self, frame):
        """
        ONNX inference — giả sử output format YOLO-like.
        Bạn có thể cần sửa phần parse output cho model cụ thể.
        """
        # Preprocess: resize + normalize
        img = cv2.resize(frame, (self.imgsz, self.imgsz))
        img = img.astype(np.float32) / 255.0
        img = img.transpose(2, 0, 1)[np.newaxis]  # BCHW
        
        outputs = self.model.run(None, {self._onnx_input_name: img})
        # Parse — mặc định YOLO output format [batch, 5+nc, 8400]
        # Sửa hàm này nếu model output format khác
        output = outputs[0]
        
        out = []
        if output.ndim == 3:  # [1, 5, 8400]
            data = output[0]  # [5, 8400]
            h, w = frame.shape[:2]
            for i in range(data.shape[1]):
                cx, cy, bw, bh = data[0,i], data[1,i], data[2,i], data[3,i]
                conf = data[4,i] if data.shape[0] > 4 else 0
                if conf < self.conf:
                    continue
                # Scale to original
                x1 = int((cx - bw/2) / self.imgsz * w)
                y1 = int((cy - bh/2) / self.imgsz * h)
                x2 = int((cx + bw/2) / self.imgsz * w)
                y2 = int((cy + bh/2) / self.imgsz * h)
                out.append({'bbox': (x1, y1, x2, y2), 'conf': float(conf)})
        return out
    
    def __repr__(self):
        return f"UniversalDetector({self.name}, type={self.model_type})"


# ═══════════════════════════════════════
# Comparison plot helpers
# ═══════════════════════════════════════

def plot_model_comparison(results_dict, metric, title, ylabel,
                          higher_is_better=True, fmt='.3f'):
    """
    Bar chart so sánh 1 metric giữa nhiều models.
    results_dict: {model_name: value}
    """
    if len(results_dict) < 2:
        return  # Không cần so sánh nếu chỉ có 1 model
    
    names = list(results_dict.keys())
    vals = list(results_dict.values())
    colors = [C['blue']] + [C['green'] if (
        (v > vals[0] and higher_is_better) or
        (v < vals[0] and not higher_is_better)
    ) else C['red'] for v in vals[1:]]
    
    fig, ax = plt.subplots(figsize=(max(8, len(names)*2.5), 4))
    bars = ax.bar(names, vals, color=colors, alpha=0.9, width=0.5)
    
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, v + max(vals)*0.02,
                f'{v:{fmt}}', ha='center', fontsize=11, fontweight='bold')
    
    # Delta labels (vs baseline)
    if len(vals) > 1:
        for i in range(1, len(vals)):
            delta = vals[i] - vals[0]
            pct = 100 * delta / max(abs(vals[0]), 1e-6)
            sign = '+' if delta > 0 else ''
            good = (delta > 0 and higher_is_better) or (delta < 0 and not higher_is_better)
            color = C['green'] if good else C['red']
            ax.text(bars[i].get_x() + bars[i].get_width()/2, max(vals)*0.01,
                    f'{sign}{pct:.1f}%', ha='center', fontsize=9,
                    color=color, fontweight='bold')
    
    ax.set_ylabel(ylabel)
    ax.set_title(title, fontweight='bold', fontsize=13)
    ax.set_ylim(0, max(vals) * 1.2)
    plt.tight_layout()
    plt.show()


def plot_latency_comparison(all_model_stats, title='Latency Comparison'):
    """
    Grouped bar: mean/p95 cho nhiều models.
    all_model_stats: {model_name: bench_stats_dict}
    """
    if len(all_model_stats) < 2:
        return
    
    names = list(all_model_stats.keys())
    means = [s['mean'] for s in all_model_stats.values()]
    p95s = [s['p95'] for s in all_model_stats.values()]
    
    x = np.arange(len(names))
    w = 0.3
    fig, ax = plt.subplots(figsize=(max(8, len(names)*2.5), 4))
    
    bars1 = ax.bar(x - w/2, means, w, label='Mean', color=C['blue'], alpha=0.9)
    bars2 = ax.bar(x + w/2, p95s, w, label='P95', color=C['yellow'], alpha=0.9)
    
    for bar, v in zip(bars1, means):
        ax.text(bar.get_x()+bar.get_width()/2, v+0.3, f'{v:.1f}', ha='center', fontsize=9)
    for bar, v in zip(bars2, p95s):
        ax.text(bar.get_x()+bar.get_width()/2, v+0.3, f'{v:.1f}', ha='center', fontsize=9)
    
    # Speedup vs baseline
    if len(means) > 1:
        for i in range(1, len(means)):
            speedup = means[0] / max(means[i], 0.01)
            color = C['green'] if speedup > 1 else C['red']
            ax.text(i, max(max(means), max(p95s)) * 1.08,
                    f'{speedup:.2f}x', ha='center', fontsize=10,
                    color=color, fontweight='bold')
    
    ax.set_xticks(x)
    ax.set_xticklabels(names, fontsize=10)
    ax.set_ylabel('Latency (ms)')
    ax.set_title(title, fontweight='bold', fontsize=13)
    ax.legend()
    plt.tight_layout()
    plt.show()


def plot_similarity_comparison(all_same, all_diff, all_thresholds, title='Embedding Quality'):
    """
    Overlay histogram same/diff cho nhiều face models.
    all_same: {model_name: [sims]}
    all_diff: {model_name: [sims]}
    all_thresholds: {model_name: (thr, acc)}
    """
    n = len(all_same)
    if n < 2:
        return
    
    fig, axes = plt.subplots(1, n, figsize=(6*n, 4), squeeze=False)
    for i, name in enumerate(all_same.keys()):
        ax = axes[0][i]
        ax.hist(all_same[name], bins=25, color=C['green'], alpha=0.7,
                label='Same', edgecolor='white', linewidth=0.3)
        ax.hist(all_diff[name], bins=25, color=C['red'], alpha=0.5,
                label='Diff', edgecolor='white', linewidth=0.3)
        thr, acc = all_thresholds[name]
        ax.axvline(thr, color='white', linestyle='--', label=f'thr={thr:.2f} acc={acc:.3f}')
        ax.set_title(name, fontweight='bold')
        ax.set_xlabel('Cosine Similarity')
        ax.legend(fontsize=7)
    
    plt.suptitle(title, fontsize=14, fontweight='bold', y=1.03)
    plt.tight_layout()
    plt.show()


print('Helpers ready (with comparison plots)')


Helpers ready


---
## 1. PlateDetector — YOLOv8n Detection
**Data**: ảnh gốc (1 folder) + YOLO labels (folder riêng)  
**Đo**: latency theo resolution, accuracy (P/R/F1 @ IoU≥0.5)

In [13]:
# ── Load tất cả plate detection models ──
plate_detectors = {}
for cfg in PLATE_DET_MODELS:
    if not os.path.exists(cfg.get('path', '')):
        print(f"❌ SKIP {cfg['name']}: {cfg.get('path','')} not found")
        continue
    print(f"Loading {cfg['name']} (type={cfg.get('type','ultralytics')})...")
    try:
        det = UniversalDetector(cfg)
        plate_detectors[cfg['name']] = det
        print(f"  ✅ {cfg['name']} loaded")
    except Exception as e:
        print(f"  ❌ {cfg['name']} FAILED: {e}")

print(f'\n{len(plate_detectors)} detector(s) ready\n')

# ── Latency benchmark ──
det_latency_all = {}

print('=== Latency (640x640 square) ===')
img_sq = np.random.randint(0, 255, (640, 640, 3), dtype=np.uint8)
for name, det in plate_detectors.items():
    s = bench(lambda d=det: d(img_sq), n=N_BENCH, label=name)
    det_latency_all[name] = s

# Real images
if HAS_DET:
    real_paths = sorted(
        glob.glob(os.path.join(DET_IMAGES_DIR, '*.jpg')) +
        glob.glob(os.path.join(DET_IMAGES_DIR, '*.png'))
    )[:50]
    loaded_imgs = [cv2.imread(p) for p in real_paths]
    loaded_imgs = [img for img in loaded_imgs if img is not None][:50]
    if loaded_imgs:
        h0, w0 = loaded_imgs[0].shape[:2]
        print(f'\n=== Latency on real images ({w0}x{h0}, padded) ===')
        for name, det in plate_detectors.items():
            idx = [0]
            def det_real(d=det):
                img = loaded_imgs[idx[0] % len(loaded_imgs)]
                idx[0] += 1
                return d(pad_to_square(img))
            bench(det_real, n=min(N_BENCH, len(loaded_imgs)*3), label=f'{name} (real)')

plot_latency_comparison(det_latency_all, 'Plate Detection — Latency Comparison')

Loading PlateDetector...
Loading /home/somethink/parking_system/models/plate_yolov8n.engine for TensorRT inference...


[03/31/2026-16:23:14] [TRT] [I] The logger passed into createInferRuntime differs from one already provided for an existing builder, runtime, or refitter. Uses of the global logger, returned by nvinfer1::getLogger(), will return the existing value.
[03/31/2026-16:23:14] [TRT] [I] Loaded engine size: 8 MiB
[03/31/2026-16:23:14] [TRT] [W] Using an engine plan file across different models of devices is not recommended and is likely to affect performance or even cause errors.
[03/31/2026-16:23:14] [TRT] [I] [MemUsageChange] TensorRT-managed allocation in IExecutionContext creation: CPU +1, GPU +15, now: CPU 2, GPU 83 (MiB)
WARNING ⚠️ Metadata not found for 'model=/home/somethink/parking_system/models/plate_yolov8n.engine'
Done.

=== Latency by Resolution ===
  Det 640x640                     avg=  21.04ms  p95=  23.55ms  fps=  47.5
  Det 640x640                     avg=  19.94ms  p95=  21.70ms  fps=  50.1


In [14]:
# ── Accuracy cho từng model ──

det_accuracy_all = {}  # {model_name: {tp, fp, fn, prec, rec, f1, ...}}

if not HAS_DET:
    print('⚠️  SKIP — chưa set DET_IMAGES_DIR / DET_LABELS_DIR')
else:
    # Load pairs 1 lần
    all_imgs = sorted(
        glob.glob(os.path.join(DET_IMAGES_DIR, '*.jpg')) +
        glob.glob(os.path.join(DET_IMAGES_DIR, '*.jpeg')) +
        glob.glob(os.path.join(DET_IMAGES_DIR, '*.png'))
    )
    pairs = []
    for img_p in all_imgs:
        lbl_p = os.path.join(DET_LABELS_DIR, Path(img_p).stem + '.txt')
        if os.path.exists(lbl_p):
            pairs.append((img_p, lbl_p))
    if len(pairs) > 500:
        random.seed(42)
        pairs = random.sample(pairs, 500)
    
    # Load ảnh 1 lần vào RAM (tránh đọc disk nhiều lần khi có nhiều models)
    pair_data = []
    for img_p, lbl_p in pairs:
        img = cv2.imread(img_p)
        if img is None: continue
        h, w = img.shape[:2]
        gt_boxes = []
        with open(lbl_p) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 5: continue
                cx,cy,bw,bh = map(float, parts[1:5])
                gt_boxes.append((
                    int((cx-bw/2)*w), int((cy-bh/2)*h),
                    int((cx+bw/2)*w), int((cy+bh/2)*h)
                ))
        pair_data.append((img, gt_boxes))
    
    print(f'{len(pair_data)} image-label pairs loaded\n')
    
    for model_name, det in plate_detectors.items():
        print(f'--- {model_name} ---')
        acc = {'tp':0, 'fp':0, 'fn':0, 'confs':[], 'times_ms':[]}
        
        for img, gt_boxes in pair_data:
            t0 = time.perf_counter()
            dets = det(pad_to_square(img))
            acc['times_ms'].append((time.perf_counter()-t0)*1000)
            
            matched = set()
            for d in sorted(dets, key=lambda d: d['conf'], reverse=True):
                acc['confs'].append(d['conf'])
                best_iou, best_gi = 0, -1
                for gi, gt in enumerate(gt_boxes):
                    if gi in matched: continue
                    v = iou_calc(d['bbox'], gt)
                    if v > best_iou: best_iou, best_gi = v, gi
                if best_iou >= 0.5 and best_gi >= 0:
                    acc['tp'] += 1; matched.add(best_gi)
                else:
                    acc['fp'] += 1
            acc['fn'] += len(gt_boxes) - len(matched)
        
        tp,fp,fn = acc['tp'], acc['fp'], acc['fn']
        prec = tp/max(tp+fp,1); rec = tp/max(tp+fn,1)
        f1 = 2*prec*rec/max(prec+rec,1e-6)
        acc.update({'prec':prec, 'rec':rec, 'f1':f1})
        det_accuracy_all[model_name] = acc
        
        print(f'  TP={tp} FP={fp} FN={fn}  P={prec:.4f} R={rec:.4f} F1={f1:.4f}')
        if acc['times_ms']:
            print(f'  Avg inference: {np.mean(acc["times_ms"]):.1f}ms')
        print()
    
    # ── So sánh accuracy ──
    if len(det_accuracy_all) >= 2:
        plot_model_comparison(
            {k: v['f1'] for k,v in det_accuracy_all.items()},
            'F1', 'Plate Detection — F1 Comparison', 'F1 Score',
            higher_is_better=True)
        plot_model_comparison(
            {k: v['prec'] for k,v in det_accuracy_all.items()},
            'Precision', 'Plate Detection — Precision', 'Precision',
            higher_is_better=True)
        plot_model_comparison(
            {k: v['rec'] for k,v in det_accuracy_all.items()},
            'Recall', 'Plate Detection — Recall', 'Recall',
            higher_is_better=True)

Found 500 image-label pairs
[03/31/2026-16:23:27] [TRT] [E] IExecutionContext::setInputShape: Error Code 3: API Usage Error (Parameter check failed, condition: engineDims.d[i] == dims.d[i]. Static dimension mismatch while setting input shape for images. Set dimensions are [1,3,416,640]. Expected dimensions are [-1,3,640,640].)
[03/31/2026-16:23:28] [TRT] [E] IExecutionContext::setInputShape: Error Code 3: API Usage Error (Parameter check failed, condition: engineDims.d[i] == dims.d[i]. Static dimension mismatch while setting input shape for images. Set dimensions are [1,3,384,640]. Expected dimensions are [-1,3,640,640].)
[03/31/2026-16:23:28] [TRT] [E] IExecutionContext::setInputShape: Error Code 3: API Usage Error (Parameter check failed, condition: engineDims.d[i] == dims.d[i]. Static dimension mismatch while setting input shape for images. Set dimensions are [1,3,416,640]. Expected dimensions are [-1,3,640,640].)
[03/31/2026-16:23:28] [TRT] [E] IExecutionContext::setInputShape: Err

KeyboardInterrupt: 

In [ ]:
# ── Detailed plots cho từng model ──
for model_name, acc in det_accuracy_all.items():
    if acc['tp'] + acc['fp'] == 0: continue
    
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    fig.suptitle(f'PlateDetector: {model_name}', fontsize=13, fontweight='bold', y=1.03)
    
    axes[0].hist(acc['confs'], bins=30, color=C['blue'], alpha=0.85, edgecolor='white', linewidth=0.3)
    axes[0].axvline(0.4, color=C['red'], linestyle='--', label='threshold=0.4')
    axes[0].set_title('Confidence'); axes[0].legend(fontsize=8)
    
    bars = axes[1].bar(['Precision','Recall','F1'],
                        [acc['prec'],acc['rec'],acc['f1']],
                        color=[C['green'],C['blue'],C['purple']], alpha=0.9)
    for b,v in zip(bars,[acc['prec'],acc['rec'],acc['f1']]):
        axes[1].text(b.get_x()+b.get_width()/2, v+0.01, f'{v:.3f}',
                     ha='center', fontsize=11, fontweight='bold')
    axes[1].set_ylim(0,1.15); axes[1].set_title('Metrics @ IoU≥0.5')
    
    axes[2].hist(acc['times_ms'], bins=30, color=C['cyan'], alpha=0.85, edgecolor='white', linewidth=0.3)
    axes[2].axvline(np.mean(acc['times_ms']), color='white', linestyle='--',
                    label=f'mean={np.mean(acc["times_ms"]):.1f}ms')
    axes[2].set_title('Latency (real)'); axes[2].set_xlabel('ms'); axes[2].legend(fontsize=8)
    
    plt.tight_layout()
    plt.show()

---
## 2. PlateOCR + Validator
**Data**: folder chứa `*.jpg` + `*.txt` cùng tên (txt chứa text biển số)  
**Đo**: exact match, edit distance, validator pass rate, latency

In [ ]:
from engine import PlateOCR
from main import PlateValidator

validator = PlateValidator()

# ── Load tất cả OCR models ──
ocr_engines = {}
for cfg in OCR_MODELS:
    print(f"Loading {cfg['name']}...")
    ocr = PlateOCR(lang=cfg['lang'], use_gpu=cfg['use_gpu'])
    ocr_engines[cfg['name']] = ocr
    print(f"  ✅ {cfg['name']} loaded")

print(f'\n{len(ocr_engines)} OCR engine(s) ready\n')

# ── Latency benchmark ──
ocr_latency_all = {}
print('=== OCR Latency (160x48 crop) ===')
crop_test = np.random.randint(0, 255, (48, 160, 3), dtype=np.uint8)
for name, ocr in ocr_engines.items():
    s = bench(lambda o=ocr: o(crop_test), n=N_BENCH, label=name)
    ocr_latency_all[name] = s

plot_latency_comparison(ocr_latency_all, 'OCR — Latency Comparison')

In [ ]:
# ── OCR Accuracy cho từng model ──

ocr_accuracy_all = {}

if not HAS_OCR:
    print('⚠️  SKIP — chưa set OCR_DATA_DIR')
else:
    ocr_imgs = sorted(
        glob.glob(os.path.join(OCR_DATA_DIR, '*.jpg')) +
        glob.glob(os.path.join(OCR_DATA_DIR, '*.jpeg')) +
        glob.glob(os.path.join(OCR_DATA_DIR, '*.png'))
    )
    pairs = []
    for img_p in ocr_imgs:
        txt_p = os.path.splitext(img_p)[0] + '.txt'
        if os.path.exists(txt_p):
            gt = open(txt_p).read().strip()
            if gt: pairs.append((img_p, gt))
    if len(pairs) > 500:
        random.seed(42); pairs = random.sample(pairs, 500)
    
    # Load crops 1 lần
    crop_data = []
    for img_p, gt in pairs:
        crop = cv2.imread(img_p)
        if crop is not None and crop.size > 0:
            crop_data.append((crop, gt, Path(img_p).name))
    
    print(f'{len(crop_data)} OCR pairs loaded\n')
    
    for model_name, ocr in ocr_engines.items():
        print(f'--- {model_name} ---')
        acc = {'exact':0, 'validated_match':0, 'validator_pass':0,
               'edit_dists':[], 'confs':[], 'times_ms':[], 'details':[]}
        
        for crop, gt_text, fname in crop_data:
            t0 = time.perf_counter()
            raw, conf = ocr(crop)
            acc['times_ms'].append((time.perf_counter()-t0)*1000)
            acc['confs'].append(conf)
            
            ocr_c = raw.replace(' ','').replace('-','').replace('.','').upper()
            gt_c = gt_text.replace(' ','').replace('-','').replace('.','').upper()
            
            exact = (ocr_c == gt_c) and len(ocr_c) > 0
            if exact: acc['exact'] += 1
            
            ocr_v = validator(raw)
            gt_v = validator(gt_text)
            if ocr_v: acc['validator_pass'] += 1
            if ocr_v and gt_v and ocr_v == gt_v: acc['validated_match'] += 1
            
            ed = edit_distance(ocr_c, gt_c)
            acc['edit_dists'].append(ed)
            acc['details'].append({'gt':gt_c, 'ocr':ocr_c, 'exact':exact, 'ed':ed, 'conf':conf, 'file':fname})
        
        N = len(acc['details'])
        acc['exact_pct'] = acc['exact']/max(N,1)
        acc['mean_ed'] = np.mean(acc['edit_dists']) if acc['edit_dists'] else 0
        ocr_accuracy_all[model_name] = acc
        
        print(f'  Exact match: {acc["exact"]}/{N} ({acc["exact_pct"]*100:.1f}%)')
        print(f'  Edit dist:   mean={acc["mean_ed"]:.2f}')
        print(f'  Latency:     {np.mean(acc["times_ms"]):.1f}ms\n')
    
    # ── So sánh ──
    if len(ocr_accuracy_all) >= 2:
        plot_model_comparison(
            {k: v['exact_pct'] for k,v in ocr_accuracy_all.items()},
            'Exact Match', 'OCR — Exact Match Rate', 'Rate',
            higher_is_better=True)
        plot_model_comparison(
            {k: v['mean_ed'] for k,v in ocr_accuracy_all.items()},
            'Edit Distance', 'OCR — Avg Edit Distance (lower=better)', 'Edit Distance',
            higher_is_better=False, fmt='.2f')

In [ ]:
# ── Detailed OCR plots per model ──
for model_name, acc in ocr_accuracy_all.items():
    if not acc['details']: continue
    N = len(acc['details'])
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle(f'PlateOCR: {model_name}', fontsize=13, fontweight='bold', y=1.02)

    # 1) Accuracy bars
    vals = [acc['exact']/max(N,1), acc['validated_match']/max(N,1), acc['validator_pass']/max(N,1)]
    bars = axes[0,0].bar(['Exact\nMatch','Validated\nMatch','Validator\nPass'],
                          vals, color=[C['green'],C['blue'],C['cyan']], alpha=0.9)
    for b,v in zip(bars,vals):
        axes[0,0].text(b.get_x()+b.get_width()/2, v+0.01, f'{v*100:.1f}%',
                       ha='center', fontsize=10, fontweight='bold')
    axes[0,0].set_ylim(0,1.15); axes[0,0].set_title('Accuracy')

    # 2) Edit distance
    ed_arr = np.array(acc['edit_dists'])
    axes[0,1].hist(ed_arr, bins=range(0, min(int(ed_arr.max())+2, 20)),
                   color=C['yellow'], alpha=0.85, edgecolor='white', linewidth=0.3)
    axes[0,1].axvline(ed_arr.mean(), color='white', linestyle='--', label=f'mean={ed_arr.mean():.2f}')
    axes[0,1].set_title('Edit Distance'); axes[0,1].legend()

    # 3) Confidence correct vs wrong
    ok_c = [d['conf'] for d in acc['details'] if d['exact']]
    bad_c = [d['conf'] for d in acc['details'] if not d['exact'] and d['conf']>0]
    if ok_c: axes[1,0].hist(ok_c, bins=20, color=C['green'], alpha=0.7,
                             label=f'Correct ({len(ok_c)})', edgecolor='white', linewidth=0.3)
    if bad_c: axes[1,0].hist(bad_c, bins=20, color=C['red'], alpha=0.5,
                              label=f'Wrong ({len(bad_c)})', edgecolor='white', linewidth=0.3)
    axes[1,0].set_title('Confidence'); axes[1,0].legend(fontsize=9)

    # 4) Latency
    t_arr = np.array(acc['times_ms'])
    axes[1,1].hist(t_arr, bins=30, color=C['purple'], alpha=0.85, edgecolor='white', linewidth=0.3)
    axes[1,1].axvline(t_arr.mean(), color='white', linestyle='--', label=f'mean={t_arr.mean():.1f}ms')
    axes[1,1].set_title('Latency'); axes[1,1].legend(fontsize=8)

    plt.tight_layout()
    plt.show()

---
## 3. FaceEngine — LFW Dataset
**Data**: LFW (Labeled Faces in the Wild) — ~13,000 ảnh mặt  
**Đo**: detection rate, same/diff person similarity, optimal threshold, latency

In [ ]:
import tarfile, urllib.request

LFW_URL = 'http://vis-www.cs.umass.edu/lfw/lfw-funneled.tgz'
LFW_TGZ = str(PROJECT_DIR / 'benchmark_data' / 'lfw-funneled.tgz')
LFW_EXTRACT = str(PROJECT_DIR / 'benchmark_data')
LFW_ROOT = os.path.join(LFW_EXTRACT, 'lfw_funneled')

os.makedirs(LFW_EXTRACT, exist_ok=True)

if os.path.isdir(LFW_ROOT) and len(os.listdir(LFW_ROOT)) > 100:
    print(f'✅ LFW exists: {LFW_ROOT}')
else:
    print('Downloading LFW (~180MB)...')
    if not os.path.exists(LFW_TGZ):
        urllib.request.urlretrieve(LFW_URL, LFW_TGZ)
    print('Extracting...')
    with tarfile.open(LFW_TGZ, 'r:gz') as tar:
        tar.extractall(LFW_EXTRACT)
    print('Done.')

lfw_images = sorted(glob.glob(os.path.join(LFW_ROOT, '*', '*.jpg')))
print(f'LFW images: {len(lfw_images)}')

# Người có ≥2 ảnh (cho same-person test)
person_imgs = defaultdict(list)
for p in lfw_images:
    person_imgs[Path(p).parent.name].append(p)
multi = {k:v for k,v in person_imgs.items() if len(v)>=2}
print(f'Persons with ≥2 images: {len(multi)}')

In [ ]:
from engine import FaceEngine

# ── Load tất cả face models ──
face_engines = {}
for cfg in FACE_MODELS:
    print(f"Loading {cfg['name']}...")
    eng = FaceEngine(model_pack=cfg['model_pack'], det_size=tuple(cfg['det_size']))
    face_engines[cfg['name']] = eng
    print(f"  ✅ {cfg['name']} loaded")

print(f'\n{len(face_engines)} face engine(s) ready\n')

# ── Latency ──
face_latency_all = {}
print('=== FaceEngine Latency (640x480) ===')
face_test = np.random.randint(0, 255, (480, 640, 3), dtype=np.uint8)
for name, eng in face_engines.items():
    s = bench(lambda e=eng: e(face_test), n=N_BENCH, label=name)
    face_latency_all[name] = s

plot_latency_comparison(face_latency_all, 'FaceEngine — Latency Comparison')

In [ ]:
# ── Detection rate cho từng model ──
N_SAMPLE = 200
sample = random.sample(lfw_images, min(N_SAMPLE, len(lfw_images)))

# Load ảnh 1 lần
sample_imgs = [(cv2.imread(p), p) for p in sample]
sample_imgs = [(img, p) for img, p in sample_imgs if img is not None]

face_det_rates = {}  # {model_name: {detected, no_face, multi, confs, times_ms}}

for model_name, eng in face_engines.items():
    print(f'--- {model_name} ---')
    acc = {'detected':0, 'no_face':0, 'multi':0, 'confs':[], 'times_ms':[]}
    
    for img, p in sample_imgs:
        t0 = time.perf_counter()
        faces = eng(img)
        acc['times_ms'].append((time.perf_counter()-t0)*1000)
        if len(faces)==0: acc['no_face']+=1
        elif len(faces)==1:
            acc['detected']+=1; acc['confs'].append(faces[0]['conf'])
        else:
            acc['multi']+=1; acc['confs'].append(max(f['conf'] for f in faces))
    
    tot = acc['detected']+acc['no_face']+acc['multi']
    acc['det_rate'] = acc['detected']/max(tot,1)
    face_det_rates[model_name] = acc
    print(f'  Detected: {acc["detected"]}/{tot} ({acc["det_rate"]*100:.1f}%)')
    print(f'  Latency:  {np.mean(acc["times_ms"]):.1f}ms\n')

if len(face_det_rates) >= 2:
    plot_model_comparison(
        {k: v['det_rate'] for k,v in face_det_rates.items()},
        'Det Rate', 'FaceEngine — Detection Rate', 'Rate',
        higher_is_better=True)

In [ ]:
# ── Same vs Different person similarity cho từng model ──
N_PAIRS = 100

all_same_sims = {}
all_diff_sims = {}
all_thresholds = {}

for model_name, eng in face_engines.items():
    print(f'--- {model_name} ---')
    
    def get_emb(path, e=eng):
        img = cv2.imread(path)
        if img is None: return None
        faces = e(img)
        return max(faces, key=lambda f: f['conf'])['embedding'] if faces else None
    
    # Same person
    same_sims = []
    for name, imgs in list(multi.items()):
        if len(same_sims) >= N_PAIRS: break
        e1, e2 = get_emb(imgs[0]), get_emb(imgs[1])
        if e1 is not None and e2 is not None:
            same_sims.append(float(np.dot(e1, e2)))
    
    # Different person
    diff_sims = []
    all_p = list(person_imgs.keys())
    for _ in range(N_PAIRS * 5):
        if len(diff_sims) >= N_PAIRS: break
        p1, p2 = random.sample(all_p, 2)
        e1, e2 = get_emb(person_imgs[p1][0]), get_emb(person_imgs[p2][0])
        if e1 is not None and e2 is not None:
            diff_sims.append(float(np.dot(e1, e2)))
    
    # Best threshold
    best_thr, best_acc = 0, 0
    for thr in np.arange(0.0, 1.0, 0.01):
        tp = sum(1 for s in same_sims if s >= thr)
        tn = sum(1 for s in diff_sims if s < thr)
        a = (tp+tn)/(len(same_sims)+len(diff_sims))
        if a > best_acc: best_acc, best_thr = a, thr
    
    all_same_sims[model_name] = same_sims
    all_diff_sims[model_name] = diff_sims
    all_thresholds[model_name] = (best_thr, best_acc)
    
    sep = np.mean(same_sims) - np.mean(diff_sims)
    print(f'  Same: mean={np.mean(same_sims):.3f}  Diff: mean={np.mean(diff_sims):.3f}')
    print(f'  Separation: {sep:.3f}  Best thr: {best_thr:.2f} (acc={best_acc:.3f})')
    print(f'  Config=0.45 {"OK ✅" if abs(best_thr-0.45)<0.15 else "KHÁC ⚠️"}\n')

# ── So sánh separation (= khả năng phân biệt người) ──
if len(all_same_sims) >= 2:
    separations = {k: np.mean(all_same_sims[k])-np.mean(all_diff_sims[k])
                   for k in all_same_sims}
    plot_model_comparison(separations, 'Separation',
                          'FaceEngine — Embedding Separation (higher=better)',
                          'Same_mean - Diff_mean', higher_is_better=True, fmt='.3f')
    
    accuracies = {k: v[1] for k,v in all_thresholds.items()}
    plot_model_comparison(accuracies, 'Best Accuracy',
                          'FaceEngine — Best Threshold Accuracy',
                          'Accuracy', higher_is_better=True)

plot_similarity_comparison(all_same_sims, all_diff_sims, all_thresholds)

In [ ]:
# ── Detailed face plots per model ──
for model_name in face_engines.keys():
    acc = face_det_rates.get(model_name, {})
    same = all_same_sims.get(model_name, [])
    diff = all_diff_sims.get(model_name, [])
    thr_info = all_thresholds.get(model_name, (0.45, 0))
    
    tot = acc.get('detected',0) + acc.get('no_face',0) + acc.get('multi',0)
    if tot == 0: continue
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle(f'FaceEngine: {model_name}', fontsize=13, fontweight='bold', y=1.02)

    # 1) Detection pie
    axes[0,0].pie([acc['detected'], acc['no_face'], acc['multi']],
                  labels=['Detected','No face','Multi'],
                  colors=[C['green'],C['red'],C['yellow']],
                  autopct='%1.1f%%', startangle=90, textprops={'fontsize':10})
    axes[0,0].set_title(f'Detection Rate (n={tot})')

    # 2) Same vs Diff
    if same and diff:
        axes[0,1].hist(same, bins=25, color=C['green'], alpha=0.7,
                       label=f'Same (n={len(same)})', edgecolor='white', linewidth=0.3)
        axes[0,1].hist(diff, bins=25, color=C['red'], alpha=0.5,
                       label=f'Diff (n={len(diff)})', edgecolor='white', linewidth=0.3)
        axes[0,1].axvline(thr_info[0], color='white', linestyle='--',
                          label=f'best={thr_info[0]:.2f}')
        axes[0,1].axvline(0.45, color=C['yellow'], linestyle=':',
                          label='config=0.45')
        axes[0,1].set_title('Same vs Diff Similarity')
        axes[0,1].set_xlabel('Cosine Similarity'); axes[0,1].legend(fontsize=8)

    # 3) Confidence
    if acc.get('confs'):
        axes[1,0].hist(acc['confs'], bins=25, color=C['blue'], alpha=0.85,
                       edgecolor='white', linewidth=0.3)
        axes[1,0].set_title('Detection Confidence')

    # 4) Latency
    if acc.get('times_ms'):
        t_arr = np.array(acc['times_ms'])
        axes[1,1].hist(t_arr, bins=30, color=C['purple'], alpha=0.85,
                       edgecolor='white', linewidth=0.3)
        axes[1,1].axvline(t_arr.mean(), color='white', linestyle='--',
                          label=f'mean={t_arr.mean():.1f}ms')
        axes[1,1].set_title('Latency (LFW)'); axes[1,1].legend(fontsize=8)

    plt.tight_layout()
    plt.show()

---
## 4. ParkingDB — pgvector (Real Embeddings from LFW)

In [ ]:
from database import ParkingDB, DIM
import yaml

with open(str(PROJECT_DIR / 'config.yaml')) as f:
    cfg = yaml.safe_load(f)
dcfg = cfg['database']
db = ParkingDB(host=dcfg['host'], port=dcfg['port'], dbname=dcfg['dbname'],
               user=dcfg['user'], password=dcfg['password'], max_cap=dcfg['max_capacity'])
print(f'DB stats: {db.stats()}')

# Dùng baseline face engine để collect real embeddings
_face_baseline = list(face_engines.values())[0]
print(f'Collecting LFW embeddings with {list(face_engines.keys())[0]}...')
real_embs = []
for p in random.sample(lfw_images, min(600, len(lfw_images))):
    img = cv2.imread(p)
    if img is None: continue
    faces = _face_baseline(img)
    if faces:
        real_embs.append(max(faces, key=lambda f:f['conf'])['embedding'])
    if len(real_embs) >= 500: break
print(f'Got {len(real_embs)} embeddings')

In [ ]:
DB_SIZES = [10, 50, 100, 200, 500]
db_search_stats, db_insert_stats = [], []

for n in DB_SIZES:
    if n > len(real_embs):
        print(f'  Skip n={n}'); continue
    
    with db._conn() as conn:
        conn.cursor().execute("DELETE FROM active WHERE plate LIKE 'BM_%'")
    
    ids = []
    t0 = time.perf_counter()
    for i in range(n):
        r = db.entry(f'BM_{i:04d}', real_embs[i], 0.9, 0.8)
        if r > 0: ids.append(r)
    t_ins = (time.perf_counter()-t0)*1000
    db_insert_stats.append({'n':n, 'total_ms':t_ins, 'per_ms':t_ins/max(len(ids),1)})
    
    q = real_embs[min(n, len(real_embs)-1)]
    s = bench(lambda: db.match_exit(q, 0.3), n=min(N_BENCH,200), warmup=3, label=f'Search n={len(ids)}')
    db_search_stats.append(s)
    
    for rid in ids: db.exit(rid, 0.5)

print('\n=== Insert ===')
for s in db_insert_stats:
    print(f'  n={s["n"]:4d}  per_record={s["per_ms"]:.2f}ms')

In [ ]:
if db_search_stats:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
    ns = [s['n'] for s in db_insert_stats]

    axes[0].plot(ns[:len(db_search_stats)], [s['mean'] for s in db_search_stats], 'o-',
                 color=C['blue'], label='Mean', linewidth=2, markersize=8)
    axes[0].plot(ns[:len(db_search_stats)], [s['p95'] for s in db_search_stats], 's--',
                 color=C['yellow'], label='P95', linewidth=2, markersize=8)
    axes[0].set_xlabel('Records'); axes[0].set_ylabel('ms')
    axes[0].set_title('Search Latency vs DB Size', fontweight='bold'); axes[0].legend()

    axes[1].bar(range(len(ns)), [s['per_ms'] for s in db_insert_stats],
                tick_label=[str(n) for n in ns], color=C['green'], alpha=0.85)
    axes[1].set_title('Insert per Record', fontweight='bold'); axes[1].set_ylabel('ms')

    last = db_search_stats[-1]
    axes[2].hist(last['raw'], bins=30, color=C['purple'], alpha=0.85, edgecolor='white', linewidth=0.3)
    axes[2].axvline(last['mean'], color='white', linestyle='--', label=f'mean={last["mean"]:.2f}ms')
    axes[2].set_title(f'Search Latency (n={ns[-1]})', fontweight='bold'); axes[2].legend(fontsize=8)

    plt.suptitle('ParkingDB — pgvector (Real Embeddings)', fontsize=14, fontweight='bold', y=1.03)
    plt.tight_layout()
    plt.savefig('/tmp/bench_db.png', dpi=120, bbox_inches='tight')
    plt.show()

---
## 5. End-to-End + Summary

In [ ]:
# ── E2E: Det → OCR → Validate → Face → DB ──
# Dùng model đầu tiên (baseline) cho mỗi component

plate_det = list(plate_detectors.values())[0] if plate_detectors else None
plate_ocr = list(ocr_engines.values())[0] if ocr_engines else None
face_eng = list(face_engines.values())[0] if face_engines else None

print(f'E2E baseline: det={list(plate_detectors.keys())[0] if plate_detectors else "NONE"}, '
      f'ocr={list(ocr_engines.keys())[0] if ocr_engines else "NONE"}, '
      f'face={list(face_engines.keys())[0] if face_engines else "NONE"}')

if HAS_DET:
    e2e_plates = sorted(glob.glob(os.path.join(DET_IMAGES_DIR,'*.jpg'))+
                        glob.glob(os.path.join(DET_IMAGES_DIR,'*.png')))[:100]
else:
    e2e_plates = None

e2e_faces = random.sample(lfw_images, min(100, len(lfw_images)))

# Seed DB
with db._conn() as conn:
    conn.cursor().execute("DELETE FROM active WHERE plate LIKE 'E2E_%'")
for i in range(min(50, len(real_embs))):
    db.entry(f'E2E_{i:03d}', real_embs[i], 0.9, 0.8)

e2e = {'plate_det':[], 'ocr':[], 'validate':[], 'face_det':[], 'quality':[], 'db_search':[], 'total':[]}
N_E2E = min(100, len(e2e_faces))

print(f'Running {N_E2E} E2E iterations...')
for i in range(N_E2E):
    tt = time.perf_counter()

    # Plate
    if e2e_plates:
        pimg = cv2.imread(e2e_plates[i % len(e2e_plates)])
    else:
        pimg = np.random.randint(0,255,(1280,1280,3), dtype=np.uint8)

    if plate_det:
        t0=time.perf_counter(); dets=plate_det(pad_to_square(pimg)); e2e['plate_det'].append((time.perf_counter()-t0)*1000)
    else:
        dets = []

    if dets and plate_ocr:
        best=max(dets,key=lambda d:d['conf'])
        x1,y1,x2,y2=best['bbox']; h,w=pimg.shape[:2]
        bw,bh=x2-x1,y2-y1; mx,my=max(10,int(bw*0.08)),max(8,int(bh*0.12))
        crop=pimg[max(0,y1-my):min(h,y2+my),max(0,x1-mx):min(w,x2+mx)]
        t0=time.perf_counter(); raw,_=plate_ocr(crop); e2e['ocr'].append((time.perf_counter()-t0)*1000)
        t0=time.perf_counter(); validator(raw); e2e['validate'].append((time.perf_counter()-t0)*1000)

    # Face
    if face_eng:
        fimg = cv2.imread(e2e_faces[i % len(e2e_faces)])
        if fimg is not None:
            t0=time.perf_counter(); faces=face_eng(fimg); e2e['face_det'].append((time.perf_counter()-t0)*1000)
            if faces:
                bf=max(faces,key=lambda f:f['conf'])
                t0=time.perf_counter(); FaceEngine.quality_score(fimg,bf['bbox']); e2e['quality'].append((time.perf_counter()-t0)*1000)
                t0=time.perf_counter(); db.match_exit(bf['embedding'],0.45); e2e['db_search'].append((time.perf_counter()-t0)*1000)

    e2e['total'].append((time.perf_counter()-tt)*1000)

with db._conn() as conn:
    conn.cursor().execute("DELETE FROM active WHERE plate LIKE 'E2E_%'")

total_avg = np.mean(e2e['total'])
print(f'\n{"Step":15s} {"Avg":>8s} {"P95":>8s} {"% total":>8s}')
print('-'*42)
for step in ['plate_det','ocr','validate','face_det','quality','db_search','total']:
    if not e2e[step]: continue
    a=np.array(e2e[step])
    pct = 100*a.mean()/total_avg if step!='total' else 100
    print(f'{step:15s} {a.mean():7.1f}ms {np.percentile(a,95):7.1f}ms {pct:7.1f}%')
print(f'\n→ E2E FPS: {1000/total_avg:.1f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

steps = ['plate_det','ocr','validate','face_det','quality','db_search']
snames = ['Plate Det','OCR','Validate','Face Det','Quality','DB Search']
scols = [C['blue'],C['green'],C['cyan'],C['purple'],C['orange'],C['pink']]
avgs = [np.mean(e2e[s]) if e2e[s] else 0 for s in steps]

left=0
for a,nm,co in zip(avgs,snames,scols):
    axes[0].barh(0, a, left=left, height=0.4, color=co, label=f'{nm}: {a:.1f}ms', alpha=0.9)
    if a>2: axes[0].text(left+a/2, 0, f'{a:.1f}', ha='center', va='center', fontsize=9, fontweight='bold')
    left+=a
axes[0].set_yticks([]); axes[0].set_xlabel('ms')
axes[0].set_title(f'Breakdown (total: {sum(avgs):.1f}ms)', fontweight='bold')
axes[0].legend(loc='upper right', fontsize=7)

nz = [(n,a,c) for n,a,c in zip(snames,avgs,scols) if a>0]
if nz:
    pn,pa,pc = zip(*nz)
    axes[1].pie(pa, labels=pn, colors=pc, autopct='%1.1f%%', startangle=90, textprops={'fontsize':9})
    axes[1].set_title('Time Distribution', fontweight='bold')

ta = np.array(e2e['total'])
axes[2].plot(ta, color=C['blue'], alpha=0.5, linewidth=0.8)
w=min(10,len(ta))
if w>1:
    r=np.convolve(ta,np.ones(w)/w,mode='valid')
    axes[2].plot(range(w-1,len(ta)),r,color=C['green'],linewidth=2,label=f'Rolling({w})')
axes[2].axhline(ta.mean(),color='white',linestyle='--',alpha=0.5,label=f'Mean: {ta.mean():.1f}ms')
axes[2].set_xlabel('Frame'); axes[2].set_ylabel('ms')
axes[2].set_title('Per-Frame Latency', fontweight='bold'); axes[2].legend(fontsize=8)

plt.suptitle('End-to-End Pipeline', fontsize=14, fontweight='bold', y=1.03)
plt.tight_layout()
plt.savefig('/tmp/bench_e2e.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── Overall Summary ──
all_stats = []

# Pick first (baseline) model from each category
if det_latency_all:
    first_det = list(det_latency_all.values())[0]
    all_stats.append(first_det)

if ocr_latency_all:
    first_ocr = list(ocr_latency_all.values())[0]
    all_stats.append(first_ocr)

if face_latency_all:
    first_face = list(face_latency_all.values())[0]
    all_stats.append(first_face)

if db_search_stats:
    all_stats.append(db_search_stats[-1])

print('='*65)
print('  SUMMARY (baseline models)')
print('='*65)
print(f'{"Component":30s} {"Mean":>8s} {"P95":>8s} {"FPS":>8s}')
print('-'*65)
for s in all_stats:
    print(f'{s["label"]:30s} {s["mean"]:7.2f}ms {s["p95"]:7.2f}ms {s["fps"]:7.1f}')
if e2e['total']:
    t=np.array(e2e['total'])
    print(f'{"End-to-End":30s} {t.mean():7.2f}ms {np.percentile(t,95):7.2f}ms {1000/t.mean():7.1f}')
print('='*65)

In [ ]:
if all_stats:
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    # Bar comparison
    labels=[s['label'] for s in all_stats]; means=[s['mean'] for s in all_stats]; p95s=[s['p95'] for s in all_stats]
    x=np.arange(len(labels)); w=0.3
    axes[0].bar(x-w/2,means,w,label='Mean',color=C['blue'],alpha=0.9)
    axes[0].bar(x+w/2,p95s,w,label='P95',color=C['yellow'],alpha=0.9)
    for i,v in enumerate(means): axes[0].text(i-w/2,v+0.3,f'{v:.1f}',ha='center',fontsize=9)
    axes[0].set_xticks(x); axes[0].set_xticklabels(labels,rotation=15,ha='right',fontsize=9)
    axes[0].set_ylabel('ms'); axes[0].set_title('Latency Comparison', fontweight='bold'); axes[0].legend()

    # Bottleneck Pareto
    total=sum(means); pcts=[m/total*100 for m in means]
    idx=np.argsort(means)[::-1]
    sl=[labels[i] for i in idx]; sm=[means[i] for i in idx]; sp=[pcts[i] for i in idx]
    bc=[C['red'] if p>40 else C['yellow'] if p>20 else C['green'] for p in sp]
    bars=axes[1].barh(sl,sm,color=bc,alpha=0.85,height=0.5)
    cum=np.cumsum(sm); ax2=axes[1].twiny()
    ax2.plot(cum/total*100,range(len(sl)),'wo-',markersize=6,linewidth=2,alpha=0.8)
    ax2.set_xlim(0,105); ax2.set_xlabel('Cumulative %',color='white',alpha=0.6)
    for b,m,p in zip(bars,sm,sp):
        axes[1].text(b.get_width()+0.3,b.get_y()+b.get_height()/2,
                     f'{m:.1f}ms ({p:.0f}%)',va='center',fontsize=10,fontweight='bold')
    axes[1].set_xlabel('ms'); axes[1].set_title('Bottleneck (Pareto)', fontweight='bold')
    axes[1].legend(handles=[
        mpatches.Patch(color=C['red'],label='>40%'),
        mpatches.Patch(color=C['yellow'],label='>20%'),
        mpatches.Patch(color=C['green'],label='<20%'),
    ], loc='lower right', fontsize=8)

    plt.suptitle('Overall Performance', fontsize=14, fontweight='bold', y=1.03)
    plt.tight_layout()
    plt.savefig('/tmp/bench_summary.png', dpi=120, bbox_inches='tight')
    plt.show()

In [ ]:
db.close()
print('\n✅ Benchmark complete!')
for f in sorted(glob.glob('/tmp/bench_*.png')):
    print(f'  {f} ({os.path.getsize(f)/1024:.0f}KB)')